In [0]:
%python
# Databricks notebook source
# PURPOSE:
# Simple guardrails + evaluation metrics table:
# - citation coverage
# - empty recommendation checks
# - invalid score checks

from pyspark.sql import functions as F

CATALOG = "vf_health"
SCHEMA = "ghana"

RECS = f"{CATALOG}.{SCHEMA}.gold_agent_recommendations"
CITES = f"{CATALOG}.{SCHEMA}.gold_citations"

EVAL = f"{CATALOG}.{SCHEMA}.gold_agent_eval"

recs = spark.table(RECS)
cites = spark.table(CITES)

# Global metrics
rec_count = recs.count()
cite_count = cites.count()
empty_rec_count = recs.filter(F.col("recommendation").isNull() | (F.trim("recommendation") == "")).count()
invalid_score_count = recs.filter((F.col("desert_score") < 0) | F.col("desert_score").isNull()).count()

rows = [{
    "metric_time": None,
    "recommendation_count": rec_count,
    "citation_count": cite_count,
    "citation_coverage_ratio": float(cite_count) / float(max(rec_count, 1)),
    "empty_recommendation_count": empty_rec_count,
    "invalid_score_count": invalid_score_count,
    "guardrail_pass": (empty_rec_count == 0 and invalid_score_count == 0)
}]

eval_df = spark.createDataFrame(rows).withColumn("metric_time", F.current_timestamp())
eval_df.write.mode("overwrite").format("delta").saveAsTable(EVAL)

display(spark.table(EVAL))